# RazorPay Track 03 — O2C / Receivables Risk Notebook

**Goal:** build the receivables part of the Revenue Recovery Agent.

This notebook deliberately separates:
1. **Observed payment behavior** — what the historical invoices actually show.
2. **Prediction** — estimate whether an invoice is likely to be paid late.
3. **Recovery decision** — later, map risk to bounded actions.

> Important: this dataset contains invoice/payment timing, but does **not** provide a clean reminder/intervention outcome history. Therefore this notebook will **not** claim causal uplift from reminders. The model is for payment-lateness risk; the recovery policy remains deterministic.


## 1. Load the dataset

The uploaded ZIP contains `dataset.csv`. Change `DATASET_PATH` if you move the data elsewhere.

In [5]:
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_PATH = "dataset.csv"

df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
display(df.head())

Shape: (50000, 19)


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,200769623,WAL-MAR corp,2/11/2020 0:00,2020,1930438491,1/26/2020,20200125,20200126,20200210,USD,RV,1,NaN,54273.28,20200126,NAH4,1.930438e+09,0
1,U001,200980828,BEN E,8/8/2019 0:00,2019,1929646410,7/22/2019,20190722,20190722,20190811,USD,RV,1,NaN,79656.60,20190722,NAD1,1.929646e+09,0
2,U001,200792734,MDV/ trust,12/30/2019 0:00,2019,1929873765,9/14/2019,20190914,20190914,20190929,USD,RV,1,NaN,2253.86,20190914,NAA8,1.929874e+09,0
3,CA02,140105686,SYSC llc,NaN,2020,2960623488,3/30/2020,20200330,20200330,20200410,CAD,RV,1,NaN,3299.70,20200331,CA10,2.960623e+09,1
4,U001,200769623,WAL-MAR foundation,11/25/2019 0:00,2019,1930147974,11/13/2019,20191113,20191113,20191128,USD,RV,1,NaN,33133.29,20191113,NAH4,1.930148e+09,0


In [6]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 rows:")
display(df.head())

Shape: (50000, 19)

Columns:
['business_code', 'cust_number', 'name_customer', 'clear_date', 'buisness_year', 'doc_id', 'posting_date', 'document_create_date', 'document_create_date.1', 'due_in_date', 'invoice_currency', 'document type', 'posting_id', 'area_business', 'total_open_amount', 'baseline_create_date', 'cust_payment_terms', 'invoice_id', 'isOpen']

Data types:


,dtype
business_code,object
cust_number,object
name_customer,object
clear_date,object
buisness_year,int64
doc_id,int64
posting_date,object
document_create_date,int64
document_create_date.1,int64
due_in_date,int64



First 5 rows:


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,200769623,WAL-MAR corp,2/11/2020 0:00,2020,1930438491,1/26/2020,20200125,20200126,20200210,USD,RV,1,NaN,54273.28,20200126,NAH4,1.930438e+09,0
1,U001,200980828,BEN E,8/8/2019 0:00,2019,1929646410,7/22/2019,20190722,20190722,20190811,USD,RV,1,NaN,79656.60,20190722,NAD1,1.929646e+09,0
2,U001,200792734,MDV/ trust,12/30/2019 0:00,2019,1929873765,9/14/2019,20190914,20190914,20190929,USD,RV,1,NaN,2253.86,20190914,NAA8,1.929874e+09,0
3,CA02,140105686,SYSC llc,NaN,2020,2960623488,3/30/2020,20200330,20200330,20200410,CAD,RV,1,NaN,3299.70,20200331,CA10,2.960623e+09,1
4,U001,200769623,WAL-MAR foundation,11/25/2019 0:00,2019,1930147974,11/13/2019,20191113,20191113,20191128,USD,RV,1,NaN,33133.29,20191113,NAH4,1.930148e+09,0


## 2. Understand the schema

We first inspect columns, missingness, and the open/closed population before creating any target.


In [2]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nOpen/closed counts:")
display(df["isOpen"].value_counts(dropna=False).rename_axis("isOpen").to_frame("count"))


Columns:
['business_code', 'cust_number', 'name_customer', 'clear_date', 'buisness_year', 'doc_id', 'posting_date', 'document_create_date', 'document_create_date.1', 'due_in_date', 'invoice_currency', 'document type', 'posting_id', 'area_business', 'total_open_amount', 'baseline_create_date', 'cust_payment_terms', 'invoice_id', 'isOpen']

Missing values:


,missing
area_business,7551
clear_date,1418
invoice_id,2
invoice_currency,1
due_in_date,1
posting_id,1
buisness_year,1
document_create_date,1
document_create_date.1,1
doc_id,1



Open/closed counts:


,count
isOpen,
0.0,6133
1.0,1417
NaN,1


## 3. Correct date parsing

`due_in_date` is stored as an integer in `YYYYMMDD` format. It must **not** be parsed as a Unix timestamp.


In [7]:
# Parse date columns correctly

df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce")
df["clear_date"] = pd.to_datetime(df["clear_date"], errors="coerce")

# due_in_date is stored as YYYYMMDD integer
df["due_date"] = pd.to_datetime(
    df["due_in_date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# These are also YYYYMMDD integers
df["document_create_date"] = pd.to_datetime(
    df["document_create_date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

df["document_create_date.1"] = pd.to_datetime(
    df["document_create_date.1"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

df["baseline_create_date"] = pd.to_datetime(
    df["baseline_create_date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("Date parsing complete.\n")

print("Date ranges:")
print("posting_date:", df["posting_date"].min(), "→", df["posting_date"].max())
print("due_date:", df["due_date"].min(), "→", df["due_date"].max())
print("clear_date:", df["clear_date"].min(), "→", df["clear_date"].max())

print("\nMissing dates:")
display(
    df[["posting_date", "due_date", "clear_date"]]
    .isna()
    .sum()
    .to_frame("missing_count")
)

Date parsing complete.

Date ranges:
posting_date: 2018-12-30 00:00:00 → 2020-05-22 00:00:00
due_date: 2018-12-24 00:00:00 → 2020-07-10 00:00:00
clear_date: 2019-01-03 00:00:00 → 2020-05-22 00:00:00

Missing dates:


,missing_count
posting_date,0
due_date,0
clear_date,10000


## 4. Closed invoices: derive the supervised target

For invoices that have already been cleared:

**days_late = clear_date − due_date**

Target:

**late_payment = 1 if days_late > 0 else 0**

This is a real observed outcome. It is not an intervention-success label.


In [8]:
# Separate closed and open invoices

closed = df[df["isOpen"] == 0].copy()
open_invoices = df[df["isOpen"] == 1].copy()

print("Closed invoices:", len(closed))
print("Open invoices:", len(open_invoices))

# Calculate days late for invoices that have actually been cleared
closed["days_late"] = (
    closed["clear_date"] - closed["due_date"]
).dt.days

print("\nDays late statistics:")
display(
    closed["days_late"].describe()
)

print("\nLate-payment distribution:")
print(
    pd.Series({
        "On time / early": (closed["days_late"] <= 0).sum(),
        "Late": (closed["days_late"] > 0).sum(),
        ">30 days late": (closed["days_late"] > 30).sum(),
        ">60 days late": (closed["days_late"] > 60).sum(),
        ">90 days late": (closed["days_late"] > 90).sum()
    })
)

Closed invoices: 40000
Open invoices: 10000

Days late statistics:


,days_late
count,40000.000000
mean,0.837700
std,10.831701
min,-89.000000
25%,-3.000000
50%,0.000000
75%,2.000000
max,204.000000



Late-payment distribution:
On time / early    23236
Late               16764
>30 days late        883
>60 days late        225
>90 days late         56
dtype: int64


## 5. Inspect the lateness distribution

This tells us whether the target has enough positive examples and whether extreme delinquency is common.


In [9]:
# Define the supervised learning target
# 1 = invoice was paid late
# 0 = invoice was paid on time or early

closed["late_payment"] = (
    closed["clear_date"] > closed["due_date"]
).astype(int)

print("Target distribution:")
print(closed["late_payment"].value_counts())

print("\nTarget proportions:")
print(
    closed["late_payment"]
    .value_counts(normalize=True)
    .sort_index()
)

Target distribution:
late_payment
0    23236
1    16764
Name: count, dtype: int64

Target proportions:
late_payment
0    0.5809
1    0.4191
Name: proportion, dtype: float64


## 6. Build initial customer history features

For each invoice in the historical closed set, we compute features describing the customer's prior payment behavior. These are designed to be strictly based on information available *before* the current invoice's due date.

In [10]:
# Sort chronologically so customer history only uses prior invoices
closed = closed.sort_values(
    ["cust_number", "due_date"]
).reset_index(drop=True)

# Previous invoices for the same customer
closed["prior_invoice_count"] = (
    closed.groupby("cust_number").cumcount()
)

# Cumulative historical late-payment rate
closed["prior_late_count"] = (
    closed.groupby("cust_number")["late_payment"]
    .cumsum()
    - closed["late_payment"]
)

closed["prior_late_rate"] = (
    closed["prior_late_count"]
    / closed["prior_invoice_count"].replace(0, np.nan)
).fillna(0)

# Previous days-late values
closed["positive_days_late"] = (
    closed["days_late"].clip(lower=0)
)

closed["prior_avg_days_late"] = (
    closed.groupby("cust_number")["positive_days_late"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

closed["prior_median_days_late"] = (
    closed.groupby("cust_number")["positive_days_late"]
    .transform(lambda x: x.shift(1).expanding().median())
).fillna(0)

# Previous invoice amounts
closed["prior_avg_invoice_amount"] = (
    closed.groupby("cust_number")["total_open_amount"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

closed["prior_max_days_late"] = (
    closed.groupby("cust_number")["positive_days_late"]
    .transform(lambda x: x.shift(1).expanding().max())
).fillna(0)

print("Customer-history features created.")

display(
    closed[
        [
            "cust_number",
            "due_date",
            "late_payment",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_median_days_late",
            "prior_avg_invoice_amount",
            "prior_max_days_late"
        ]
    ].head(15)
)

Customer-history features created.


,cust_number,due_date,late_payment,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late
0,100000048,2019-09-25,1,0,0.000000,0.000000,0.0,0.000000,0.0
1,100000048,2019-11-17,1,1,1.000000,30.000000,30.0,2301.390000,30.0
2,100000051,2019-02-08,0,0,0.000000,0.000000,0.0,0.000000,0.0
3,100000051,2019-02-25,0,1,0.000000,0.000000,0.0,35305.100000,0.0
4,100000051,2019-10-09,1,2,0.000000,0.000000,0.0,32356.650000,0.0
5,100000158,2019-04-14,1,0,0.000000,0.000000,0.0,0.000000,0.0
6,100000158,2019-05-14,1,1,1.000000,38.000000,38.0,50299.350000,38.0
7,100000158,2019-05-26,1,2,1.000000,34.000000,34.0,68406.915000,38.0
8,100000158,2019-06-30,0,3,1.000000,32.666667,30.0,68991.190000,38.0
9,100000158,2019-09-10,1,4,0.750000,24.500000,30.0,51841.557500,38.0


## 7. Refine leakage-free customer history

To ensure no data leakage, customer history features must use only **prior cleared invoices whose due date is strictly earlier than the current invoice's due date**. Invoices with the same `cust_number` and `due_date` must not contribute to each other's history.

We refine the history features to aggregate information at the customer-due-date level and then compute cumulative statistics based on strictly prior due dates.

In [11]:
# Check for multiple invoices from the same customer
# sharing the exact same due date.

same_day = (
    closed.groupby(["cust_number", "due_date"])
    .size()
    .reset_index(name="invoice_count")
)

same_day = same_day[same_day["invoice_count"] > 1]

print("Customer/date groups with multiple invoices:", len(same_day))
print("Invoices involved:", same_day["invoice_count"].sum())

print("\nLargest groups:")
display(
    same_day.sort_values("invoice_count", ascending=False).head(10)
)

Customer/date groups with multiple invoices: 5963
Invoices involved: 26053

Largest groups:


,cust_number,due_date,invoice_count
13748,200769623,2019-12-23,58
13753,200769623,2019-12-28,44
13678,200769623,2019-10-14,44
13736,200769623,2019-12-11,43
13405,200769623,2019-01-14,41
13614,200769623,2019-08-11,40
13634,200769623,2019-08-31,39
13557,200769623,2019-06-15,39
13536,200769623,2019-05-25,38
13750,200769623,2019-12-25,38


In [12]:
# Rebuild customer history strictly from invoices
# with an EARLIER due date.
#
# Invoices sharing the same customer + due date must NOT
# contribute to each other's historical features.

# Aggregate historical information at customer + due_date level
daily_customer = (
    closed
    .groupby(["cust_number", "due_date"], as_index=False)
    .agg(
        invoice_count=("late_payment", "size"),
        late_count=("late_payment", "sum"),
        avg_days_late=("positive_days_late", "mean"),
        median_days_late=("positive_days_late", "median"),
        avg_invoice_amount=("total_open_amount", "mean"),
        max_days_late=("positive_days_late", "max")
    )
    .sort_values(["cust_number", "due_date"])
)

# Cumulative history BEFORE the current due-date bucket
g = daily_customer.groupby("cust_number")

daily_customer["prior_invoice_count"] = (
    g["invoice_count"].cumsum() - daily_customer["invoice_count"]
)

daily_customer["prior_late_count"] = (
    g["late_count"].cumsum() - daily_customer["late_count"]
)

daily_customer["prior_late_rate"] = (
    daily_customer["prior_late_count"]
    / daily_customer["prior_invoice_count"].replace(0, np.nan)
).fillna(0)

# For averages/medians/max, use shifted cumulative values.
daily_customer["prior_avg_days_late"] = (
    g["avg_days_late"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

daily_customer["prior_median_days_late"] = (
    g["median_days_late"]
    .transform(lambda x: x.shift(1).expanding().median())
).fillna(0)

daily_customer["prior_avg_invoice_amount"] = (
    g["avg_invoice_amount"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

daily_customer["prior_max_days_late"] = (
    g["max_days_late"]
    .transform(lambda x: x.shift(1).expanding().max())
).fillna(0)

# Keep only the history columns needed for the invoice-level dataset
history_cols = [
    "cust_number",
    "due_date",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

history = daily_customer[history_cols]

# Remove the old history columns from closed
old_history_cols = [
    "prior_invoice_count",
    "prior_late_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

closed = closed.drop(columns=old_history_cols, errors="ignore")

# Merge the strictly-prior history back onto each invoice
closed = closed.merge(
    history,
    on=["cust_number", "due_date"],
    how="left",
    validate="many_to_one"
)

print("Strictly-prior customer history rebuilt.")

display(
    closed[
        [
            "cust_number",
            "due_date",
            "late_payment",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_median_days_late",
            "prior_avg_invoice_amount",
            "prior_max_days_late"
        ]
    ].head(15)
)

Strictly-prior customer history rebuilt.


,cust_number,due_date,late_payment,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late
0,100000048,2019-09-25,1,0,0.000000,0.000000,0.0,0.0000,0.0
1,100000048,2019-11-17,1,1,1.000000,30.000000,30.0,2301.3900,30.0
2,100000051,2019-02-08,0,0,0.000000,0.000000,0.0,0.0000,0.0
3,100000051,2019-02-25,0,1,0.000000,0.000000,0.0,35305.1000,0.0
4,100000051,2019-10-09,1,2,0.000000,0.000000,0.0,32356.6500,0.0
5,100000158,2019-04-14,1,0,0.000000,0.000000,0.0,0.0000,0.0
6,100000158,2019-05-14,1,1,1.000000,38.000000,38.0,50299.3500,38.0
7,100000158,2019-05-26,1,2,1.000000,34.000000,34.0,68406.9150,38.0
8,100000158,2019-06-30,0,3,1.000000,32.666667,30.0,68991.1900,38.0
9,100000158,2019-09-10,1,4,0.750000,24.500000,30.0,51841.5575,38.0


## 8. Key modeling rule

The prediction point is **the invoice due date**.

At that point, we can know:
- invoice amount
- payment terms
- currency/business attributes
- customer history available before this invoice's due date

We cannot know:
- `clear_date`
- `days_late`
- future settlement behavior

The next step is to construct the same leakage-free feature logic for historical closed invoices, then use a chronological train/test split.


## 9. What we have established so far

The current 50k-invoice dataset gives us a substantially stronger O2C foundation than the abandoned reminder dataset:

- enough invoices for supervised learning
- observed settlement dates
- observed due dates
- invoice amounts
- customer IDs
- enough history to construct customer payment behavior

But it **does not establish reminder effectiveness**. Therefore the eventual agent should say:

> “This invoice has high predicted late-payment risk; according to the recovery policy, take action X.”

It should **not** say:

> “Reminder X will recover ₹Y,”

unless we obtain intervention/outcome data that supports that causal claim.


In [13]:
# Current-invoice features

closed["invoice_amount"] = closed["total_open_amount"]

closed["posting_to_due_days"] = (
    closed["due_date"] - closed["posting_date"]
).dt.days

closed["due_month"] = closed["due_date"].dt.month
closed["due_day_of_week"] = closed["due_date"].dt.dayofweek

closed["posting_month"] = closed["posting_date"].dt.month

# Confirm the feature set
feature_columns = [
    "invoice_amount",
    "posting_to_due_days",
    "due_month",
    "due_day_of_week",
    "posting_month",
    "business_code",
    "invoice_currency",
    "cust_payment_terms",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

print("Feature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

print("\nMissing values in selected features:")
display(
    closed[feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("\nFeature preview:")
display(
    closed[feature_columns + ["late_payment"]].head()
)

Feature columns:
 1. invoice_amount
 2. posting_to_due_days
 3. due_month
 4. due_day_of_week
 5. posting_month
 6. business_code
 7. invoice_currency
 8. cust_payment_terms
 9. prior_invoice_count
10. prior_late_rate
11. prior_avg_days_late
12. prior_median_days_late
13. prior_avg_invoice_amount
14. prior_max_days_late

Missing values in selected features:


,missing_count
invoice_amount,0
posting_to_due_days,0
due_month,0
due_day_of_week,0
posting_month,0
business_code,0
invoice_currency,0
cust_payment_terms,0
prior_invoice_count,0
prior_late_rate,0



Feature preview:


,invoice_amount,posting_to_due_days,due_month,due_day_of_week,posting_month,business_code,invoice_currency,cust_payment_terms,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late,late_payment
0,2301.39,90,9,2,6,U001,USD,NAVM,0,0.0,0.0,0.0,0.00,0.0,1
1,122220.28,90,11,6,8,U001,USD,NAVM,1,1.0,30.0,30.0,2301.39,30.0,1
2,35305.10,30,2,4,1,U001,USD,NAD5,0,0.0,0.0,0.0,0.00,0.0,0
3,29408.20,30,2,0,1,U001,USD,NAD5,1,0.0,0.0,0.0,35305.10,0.0,0
4,37169.00,30,10,2,9,U001,USD,NAD5,2,0.0,0.0,0.0,32356.65,0.0,1


In [14]:
# Chronological train/test split
# 80% earliest invoices -> training
# 20% latest invoices -> testing

model_data = closed.sort_values("due_date").reset_index(drop=True)

split_idx = int(len(model_data) * 0.80)

train = model_data.iloc[:split_idx].copy()
test = model_data.iloc[split_idx:].copy()

X_train = train[feature_columns].copy()
y_train = train["late_payment"].copy()

X_test = test[feature_columns].copy()
y_test = test["late_payment"].copy()

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining period:")
print(train["due_date"].min(), "→", train["due_date"].max())

print("\nTest period:")
print(test["due_date"].min(), "→", test["due_date"].max())

print("\nTraining target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Training set: (32000, 14)
Test set: (8000, 14)

Training period:
2018-12-24 00:00:00 → 2019-12-04 00:00:00

Test period:
2019-12-04 00:00:00 → 2020-06-07 00:00:00

Training target rate: 0.425625
Test target rate: 0.393


In [15]:
# Use a strict date cutoff so no calendar date appears
# in both training and testing.

cutoff_date = model_data["due_date"].quantile(0.80)

train = model_data[model_data["due_date"] < cutoff_date].copy()
test = model_data[model_data["due_date"] >= cutoff_date].copy()

X_train = train[feature_columns].copy()
y_train = train["late_payment"].copy()

X_test = test[feature_columns].copy()
y_test = test["late_payment"].copy()

print("Cutoff date:", cutoff_date)

print("\nTraining set:", X_train.shape)
print("Training period:", train["due_date"].min(), "→", train["due_date"].max())
print("Training target rate:", y_train.mean())

print("\nTest set:", X_test.shape)
print("Test period:", test["due_date"].min(), "→", test["due_date"].max())
print("Test target rate:", y_test.mean())

print(
    "\nDate overlap:",
    set(train["due_date"].dt.date).intersection(
        set(test["due_date"].dt.date)
    )
)

Cutoff date: 2019-12-04 00:00:00

Training set: (31909, 14)
Training period: 2018-12-24 00:00:00 → 2019-12-03 00:00:00
Training target rate: 0.42614936224889527

Test set: (8091, 14)
Test period: 2019-12-04 00:00:00 → 2020-06-07 00:00:00
Test target rate: 0.3912989741688296

Date overlap: set()


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Separate numerical and categorical features
categorical_features = [
    "business_code",
    "invoice_currency",
    "cust_payment_terms"
]

numerical_features = [
    "invoice_amount",
    "posting_to_due_days",
    "due_month",
    "due_day_of_week",
    "posting_month",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=180,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

o2c_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

print("Training O2C late-payment model...")

o2c_model.fit(X_train, y_train)

print("Training complete.")

Training O2C late-payment model...
Training complete.


In [18]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    brier_score_loss
)

# Predict probabilities for the untouched chronological test set
y_test_prob = o2c_model.predict_proba(X_test)[:, 1]

# Default 0.50 classification threshold
y_test_pred = (y_test_prob >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_test_prob)
pr_auc = average_precision_score(y_test, y_test_prob)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
brier = brier_score_loss(y_test, y_test_prob)

print("O2C MODEL — CHRONOLOGICAL TEST RESULTS")
print("----------------------------------------")
print(f"ROC-AUC:       {roc_auc:.4f}")
print(f"PR-AUC:        {pr_auc:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"Brier score:   {brier:.4f}")
print(f"Test prevalence: {y_test.mean():.4f}")

O2C MODEL — CHRONOLOGICAL TEST RESULTS
----------------------------------------
ROC-AUC:       0.8264
PR-AUC:        0.7743
Precision:     0.7185
Recall:        0.7230
Brier score:   0.1635
Test prevalence: 0.3913


In [19]:
# Top-10% risk concentration

test_ranked = test.copy()
test_ranked["predicted_late_probability"] = y_test_prob

test_ranked = test_ranked.sort_values(
    "predicted_late_probability",
    ascending=False
).reset_index(drop=True)

top_10_count = int(len(test_ranked) * 0.10)

top_10 = test_ranked.iloc[:top_10_count]

top_10_late_rate = top_10["late_payment"].mean()
overall_late_rate = test_ranked["late_payment"].mean()

top_10_late_cases = top_10["late_payment"].sum()
total_late_cases = test_ranked["late_payment"].sum()

lift = top_10_late_rate / overall_late_rate
capture_rate = top_10_late_cases / total_late_cases

print("TOP-10% RISK CONCENTRATION")
print("--------------------------")
print(f"Top-10% invoices:       {len(top_10)}")
print(f"Late rate in top-10%:   {top_10_late_rate:.4f}")
print(f"Overall late rate:      {overall_late_rate:.4f}")
print(f"Lift:                   {lift:.4f}x")
print(f"Late cases captured:    {top_10_late_cases:.0f}")
print(f"Total late cases:       {total_late_cases:.0f}")
print(f"Capture rate:           {capture_rate:.4f}")

TOP-10% RISK CONCENTRATION
--------------------------
Top-10% invoices:       809
Late rate in top-10%:   0.8900
Overall late rate:      0.3913
Lift:                   2.2744x
Late cases captured:    720
Total late cases:       3166
Capture rate:           0.2274


In [20]:
# Inspect the highest-risk invoices

display(
    test_ranked[
        [
            "cust_number",
            "invoice_amount",
            "due_date",
            "cust_payment_terms",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_max_days_late",
            "late_payment",
            "predicted_late_probability"
        ]
    ].head(20)
)

,cust_number,invoice_amount,due_date,cust_payment_terms,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_max_days_late,late_payment,predicted_late_probability
0,140106408,83057.35,2019-12-11,CA10,371,1.0,9.993909,120.0,1,0.980642
1,140106408,39793.00,2019-12-08,CA10,365,1.0,10.030756,120.0,1,0.978595
2,140106408,18036.47,2019-12-08,CA10,365,1.0,10.030756,120.0,1,0.978595
3,140106408,3809.08,2019-12-08,CA10,365,1.0,10.030756,120.0,1,0.978240
4,140106408,152182.58,2019-12-19,CA10,378,1.0,9.924752,120.0,1,0.977719
5,140106408,66907.76,2019-12-11,CA10,371,1.0,9.993909,120.0,1,0.976815
6,140106408,26785.97,2019-12-12,CA10,373,1.0,9.976263,120.0,1,0.976605
7,140106408,7214.60,2019-12-10,CA10,369,1.0,10.021939,120.0,1,0.976540
8,140106408,2636.39,2019-12-29,CA10,385,1.0,9.829126,120.0,1,0.976125
9,CCU013,14861.16,2020-01-03,NAX2,464,1.0,43.930279,204.0,1,0.976121


In [21]:
# Check whether invoice_id uniquely identifies records

print("Total test rows:", len(test))
print("Unique invoice IDs:", test["invoice_id"].nunique())

duplicate_invoice_ids = (
    test.groupby("invoice_id")
    .size()
    .reset_index(name="row_count")
)

duplicate_invoice_ids = duplicate_invoice_ids[
    duplicate_invoice_ids["row_count"] > 1
]

print("\nInvoice IDs appearing more than once:", len(duplicate_invoice_ids))
print(
    "Rows belonging to duplicated invoice IDs:",
    duplicate_invoice_ids["row_count"].sum()
)

print("\nExamples:")
display(
    duplicate_invoice_ids
    .sort_values("row_count", ascending=False)
    .head(10)
)

Total test rows: 8091
Unique invoice IDs: 7942

Invoice IDs appearing more than once: 146
Rows belonging to duplicated invoice IDs: 295

Examples:


,invoice_id,row_count
403,1.930179e+09,3
3625,1.930392e+09,3
5387,1.930476e+09,3
301,1.930173e+09,2
219,1.930169e+09,2
315,1.930174e+09,2
337,1.930174e+09,2
383,1.930178e+09,2
328,1.930174e+09,2
409,1.930179e+09,2


In [23]:
# Inspect one duplicated invoice ID

example_id = duplicate_invoice_ids.iloc[0]["invoice_id"]

print("Inspecting invoice_id:", example_id)

display(
    test_ranked[test_ranked["invoice_id"] == example_id][
        [
            "invoice_id",
            "doc_id",
            "cust_number",
            "name_customer",
            "posting_date",
            "due_date",
            "clear_date",
            "invoice_amount",
            "cust_payment_terms",
            "late_payment",
            "predicted_late_probability"
        ]
    ]
)

Inspecting invoice_id: 1930162255.0


,invoice_id,doc_id,cust_number,name_customer,posting_date,due_date,clear_date,invoice_amount,cust_payment_terms,late_payment,predicted_late_probability
3105,1.930162e+09,1930162255,200418007,AM,2019-11-20,2019-12-05,2019-12-05,49150.97,NAA8,0,0.518447
3106,1.930162e+09,1930162255,200418007,AM,2019-11-20,2019-12-05,2019-12-05,49150.97,NAA8,0,0.518447


In [24]:
# Check exact duplicate records

exact_duplicates = df.duplicated(keep=False)

print("Total rows:", len(df))
print("Exact duplicate rows:", exact_duplicates.sum())
print("Percentage duplicated:", f"{exact_duplicates.mean() * 100:.2f}%")

print("\nExample exact duplicates:")
display(
    df[exact_duplicates]
    .sort_values("invoice_id")
    .head(10)
)

Total rows: 50000
Exact duplicate rows: 2308
Percentage duplicated: 4.62%

Example exact duplicates:


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen,due_date
37468,U001,200794332,COST foundation,2019-01-14,2019,1928543057,2018-12-31,2018-12-30,2018-12-31,20190115,USD,RV,1,NaN,15798.14,2018-12-31,NAAX,1.928543e+09,0,2019-01-15
7057,U001,200794332,COST foundation,2019-01-14,2019,1928543057,2018-12-31,2018-12-30,2018-12-31,20190115,USD,RV,1,NaN,15798.14,2018-12-31,NAAX,1.928543e+09,0,2019-01-15
45187,U001,200769623,WAL-MAR systems,2019-01-14,2019,1928543373,2018-12-31,2018-12-30,2018-12-31,20190115,USD,RV,1,NaN,21168.64,2018-12-31,NAH4,1.928543e+09,0,2019-01-15
49361,U001,200769623,WAL-MAR systems,2019-01-14,2019,1928543373,2018-12-31,2018-12-30,2018-12-31,20190115,USD,RV,1,NaN,21168.64,2018-12-31,NAH4,1.928543e+09,0,2019-01-15
47206,U001,200769623,WAL-MAR in,2019-01-14,2019,1928545098,2018-12-31,2018-12-31,2018-12-31,20190115,USD,RV,1,NaN,43486.59,2018-12-31,NAH4,1.928545e+09,0,2019-01-15
46651,U001,200769623,WAL-MAR in,2019-01-14,2019,1928545098,2018-12-31,2018-12-31,2018-12-31,20190115,USD,RV,1,NaN,43486.59,2018-12-31,NAH4,1.928545e+09,0,2019-01-15
46064,U001,200707822,PUBLI co,2019-01-22,2019,1928551864,2019-01-03,2019-01-02,2019-01-03,20190118,USD,RV,1,NaN,81475.45,2019-01-03,NAA8,1.928552e+09,0,2019-01-18
44305,U001,200707822,PUBLI co,2019-01-22,2019,1928551864,2019-01-03,2019-01-02,2019-01-03,20190118,USD,RV,1,NaN,81475.45,2019-01-03,NAA8,1.928552e+09,0,2019-01-18
38983,U001,100028210,WEST,2019-01-28,2019,1928553960,2019-01-02,2019-01-02,2019-01-02,20190117,USD,RV,1,NaN,9813.67,2019-01-02,NAA8,1.928554e+09,0,2019-01-17
15750,U001,100028210,WEST,2019-01-28,2019,1928553960,2019-01-02,2019-01-02,2019-01-02,20190117,USD,RV,1,NaN,9813.67,2019-01-02,NAA8,1.928554e+09,0,2019-01-17


In [25]:
# Remove exact duplicate records.
# Keep the first occurrence of each identical row.

before = len(df)

df_clean = df.drop_duplicates().copy()

after = len(df_clean)

print("Rows before deduplication:", before)
print("Rows after deduplication:", after)
print("Exact duplicates removed:", before - after)
print(
    "Percentage removed:",
    f"{((before - after) / before) * 100:.2f}%"
)

print("\nRemaining exact duplicates:")
print(df_clean.duplicated().sum())

Rows before deduplication: 50000
Rows after deduplication: 48839
Exact duplicates removed: 1161
Percentage removed: 2.32%

Remaining exact duplicates:
0


In [26]:
# Rebuild closed/open datasets from the deduplicated data

closed = df_clean[df_clean["isOpen"] == 0].copy()
open_invoices = df_clean[df_clean["isOpen"] == 1].copy()

print("Clean dataset:", len(df_clean))
print("Closed invoices:", len(closed))
print("Open invoices:", len(open_invoices))

print("\nOpen/closed check:")
print(df_clean["isOpen"].value_counts())

print("\nMissing clear dates:")
print(df_clean["clear_date"].isna().sum())

Clean dataset: 48839
Closed invoices: 39158
Open invoices: 9681

Open/closed check:
isOpen
0    39158
1     9681
Name: count, dtype: int64

Missing clear dates:
9681


In [27]:
# Recalculate lateness and target on the deduplicated dataset

closed["days_late"] = (
    closed["clear_date"] - closed["due_date"]
).dt.days

closed["late_payment"] = (
    closed["clear_date"] > closed["due_date"]
).astype(int)

print("Closed invoices:", len(closed))

print("\nDays late statistics:")
display(closed["days_late"].describe())

print("\nTarget distribution:")
print(closed["late_payment"].value_counts())

print("\nTarget proportions:")
print(
    closed["late_payment"]
    .value_counts(normalize=True)
    .sort_index()
)

Closed invoices: 39158

Days late statistics:


,days_late
count,39158.000000
mean,0.837147
std,10.843083
min,-89.000000
25%,-3.000000
50%,0.000000
75%,2.000000
max,204.000000



Target distribution:
late_payment
0    22740
1    16418
Name: count, dtype: int64

Target proportions:
late_payment
0    0.580724
1    0.419276
Name: proportion, dtype: float64


In [28]:
# Rebuild leakage-free customer history on the deduplicated dataset

closed = closed.sort_values(
    ["cust_number", "due_date"]
).reset_index(drop=True)

# Aggregate invoices by customer + due date.
# This prevents invoices on the same due date from seeing
# each other's outcomes.

daily_customer = (
    closed
    .groupby(["cust_number", "due_date"], as_index=False)
    .agg(
        invoice_count=("late_payment", "size"),
        late_count=("late_payment", "sum"),
        avg_days_late=("days_late", lambda x: x.clip(lower=0).mean()),
        median_days_late=("days_late", lambda x: x.clip(lower=0).median()),
        avg_invoice_amount=("total_open_amount", "mean"),
        max_days_late=("days_late", lambda x: x.clip(lower=0).max())
    )
    .sort_values(["cust_number", "due_date"])
)

g = daily_customer.groupby("cust_number")

# History from strictly earlier due-date groups
daily_customer["prior_invoice_count"] = (
    g["invoice_count"].cumsum()
    - daily_customer["invoice_count"]
)

daily_customer["prior_late_count"] = (
    g["late_count"].cumsum()
    - daily_customer["late_count"]
)

daily_customer["prior_late_rate"] = (
    daily_customer["prior_late_count"]
    / daily_customer["prior_invoice_count"].replace(0, np.nan)
).fillna(0)

daily_customer["prior_avg_days_late"] = (
    g["avg_days_late"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

daily_customer["prior_median_days_late"] = (
    g["median_days_late"]
    .transform(lambda x: x.shift(1).expanding().median())
).fillna(0)

daily_customer["prior_avg_invoice_amount"] = (
    g["avg_invoice_amount"]
    .transform(lambda x: x.shift(1).expanding().mean())
).fillna(0)

daily_customer["prior_max_days_late"] = (
    g["max_days_late"]
    .transform(lambda x: x.shift(1).expanding().max())
).fillna(0)

history_cols = [
    "cust_number",
    "due_date",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

history = daily_customer[history_cols]

closed = closed.merge(
    history,
    on=["cust_number", "due_date"],
    how="left",
    validate="many_to_one"
)

print("Strictly-prior history rebuilt on clean data.")

display(
    closed[
        [
            "cust_number",
            "due_date",
            "late_payment",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_median_days_late",
            "prior_avg_invoice_amount",
            "prior_max_days_late"
        ]
    ].head(15)
)

Strictly-prior history rebuilt on clean data.


,cust_number,due_date,late_payment,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late
0,100000048,2019-09-25,1,0,0.00,0.000000,0.0,0.0000,0.0
1,100000048,2019-11-17,1,1,1.00,30.000000,30.0,2301.3900,30.0
2,100000051,2019-02-08,0,0,0.00,0.000000,0.0,0.0000,0.0
3,100000051,2019-02-25,0,1,0.00,0.000000,0.0,35305.1000,0.0
4,100000051,2019-10-09,1,2,0.00,0.000000,0.0,32356.6500,0.0
5,100000158,2019-04-14,1,0,0.00,0.000000,0.0,0.0000,0.0
6,100000158,2019-05-14,1,1,1.00,38.000000,38.0,50299.3500,38.0
7,100000158,2019-05-26,1,2,1.00,34.000000,34.0,68406.9150,38.0
8,100000158,2019-06-30,0,3,1.00,32.666667,30.0,68991.1900,38.0
9,100000158,2019-09-10,1,4,0.75,24.500000,30.0,51841.5575,38.0


In [29]:
# Build final O2C model dataset from clean, leakage-free features

closed["invoice_amount"] = closed["total_open_amount"]

closed["posting_to_due_days"] = (
    closed["due_date"] - closed["posting_date"]
).dt.days

closed["due_month"] = closed["due_date"].dt.month
closed["due_day_of_week"] = closed["due_date"].dt.dayofweek
closed["posting_month"] = closed["posting_date"].dt.month

feature_cols = [
    "invoice_amount",
    "posting_to_due_days",
    "due_month",
    "due_day_of_week",
    "posting_month",
    "business_code",
    "invoice_currency",
    "cust_payment_terms",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

target_col = "late_payment"

model_data = closed[
    ["due_date"] + feature_cols + [target_col]
].copy()

print("Model rows:", len(model_data))
print("Features:", len(feature_cols))

print("\nMissing values:")
print(model_data[feature_cols].isna().sum())

print("\nFeature preview:")
display(model_data[feature_cols + [target_col]].head())

Model rows: 39158
Features: 14

Missing values:
invoice_amount              0
posting_to_due_days         0
due_month                   0
due_day_of_week             0
posting_month               0
business_code               0
invoice_currency            0
cust_payment_terms          0
prior_invoice_count         0
prior_late_rate             0
prior_avg_days_late         0
prior_median_days_late      0
prior_avg_invoice_amount    0
prior_max_days_late         0
dtype: int64

Feature preview:


,invoice_amount,posting_to_due_days,due_month,due_day_of_week,posting_month,business_code,invoice_currency,cust_payment_terms,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late,late_payment
0,2301.39,90,9,2,6,U001,USD,NAVM,0,0.0,0.0,0.0,0.00,0.0,1
1,122220.28,90,11,6,8,U001,USD,NAVM,1,1.0,30.0,30.0,2301.39,30.0,1
2,35305.10,30,2,4,1,U001,USD,NAD5,0,0.0,0.0,0.0,0.00,0.0,0
3,29408.20,30,2,0,1,U001,USD,NAD5,1,0.0,0.0,0.0,35305.10,0.0,0
4,37169.00,30,10,2,9,U001,USD,NAD5,2,0.0,0.0,0.0,32356.65,0.0,1


In [31]:
# Strict chronological train/test split

cutoff_date = model_data["due_date"].quantile(0.80)

train = model_data[
    model_data["due_date"] < cutoff_date
].copy()

test = model_data[
    model_data["due_date"] >= cutoff_date
].copy()

X_train = train[feature_cols]
y_train = train[target_col]

X_test = test[feature_cols]
y_test = test[target_col]

print("Cutoff date:", cutoff_date)

print("\nTrain:")
print("Rows:", len(train))
print("Period:", train["due_date"].min(), "to", train["due_date"].max())
print("Late-payment rate:", y_train.mean())

print("\nTest:")
print("Rows:", len(test))
print("Period:", test["due_date"].min(), "to", test["due_date"].max())
print("Late-payment rate:", y_test.mean())

print("\nDate overlap:")
print(set(train["due_date"]).intersection(set(test["due_date"])))

Cutoff date: 2019-12-05 00:00:00

Train:
Rows: 31315
Period: 2018-12-24 00:00:00 to 2019-12-04 00:00:00
Late-payment rate: 0.4257384639948906

Test:
Rows: 7843
Period: 2019-12-05 00:00:00 to 2020-06-07 00:00:00
Late-payment rate: 0.39347188575800074

Date overlap:
set()


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

categorical_features = [
    "business_code",
    "invoice_currency",
    "cust_payment_terms"
]

numerical_features = [
    "invoice_amount",
    "posting_to_due_days",
    "due_month",
    "due_day_of_week",
    "posting_month",
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=180,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

o2c_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

o2c_model.fit(X_train, y_train)

print("Final clean O2C model trained.")

Final clean O2C model trained.


In [33]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    brier_score_loss
)

# Predict probability of late payment
y_prob = o2c_model.predict_proba(X_test)[:, 1]

# Classification at 0.50 threshold
y_pred = (y_prob >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_prob)

print("Final Clean O2C Model")
print("---------------------")
print(f"ROC-AUC:          {roc_auc:.4f}")
print(f"PR-AUC:           {pr_auc:.4f}")
print(f"Precision @ 0.50: {precision:.4f}")
print(f"Recall @ 0.50:    {recall:.4f}")
print(f"Brier Score:      {brier:.4f}")

Final Clean O2C Model
---------------------
ROC-AUC:          0.8282
PR-AUC:           0.7780
Precision @ 0.50: 0.7257
Recall @ 0.50:    0.7194
Brier Score:      0.1624


In [34]:
# Top-10% risk concentration on the final clean test set

test_ranked = test.copy()
test_ranked["late_payment_probability"] = y_prob

# Rank highest-risk invoices first
test_ranked = test_ranked.sort_values(
    "late_payment_probability",
    ascending=False
).reset_index(drop=True)

top_10_n = int(len(test_ranked) * 0.10)

top_10 = test_ranked.iloc[:top_10_n]

overall_late_rate = test_ranked["late_payment"].mean()
top_10_late_rate = top_10["late_payment"].mean()

lift = top_10_late_rate / overall_late_rate

total_late = test_ranked["late_payment"].sum()
late_in_top_10 = top_10["late_payment"].sum()

capture_rate = late_in_top_10 / total_late

print("Top-10% Risk Concentration")
print("--------------------------")
print(f"Test invoices:          {len(test_ranked)}")
print(f"Top-10% invoices:       {len(top_10)}")
print(f"Overall late rate:      {overall_late_rate:.4f}")
print(f"Top-10% late rate:      {top_10_late_rate:.4f}")
print(f"Lift:                   {lift:.4f}x")
print(f"Total late invoices:    {total_late}")
print(f"Late invoices captured: {late_in_top_10}")
print(f"Capture rate:           {capture_rate:.4f}")

Top-10% Risk Concentration
--------------------------
Test invoices:          7843
Top-10% invoices:       784
Overall late rate:      0.3935
Top-10% late rate:      0.8916
Lift:                   2.2659x
Total late invoices:    3086
Late invoices captured: 699
Capture rate:           0.2265


In [35]:
# Prepare open invoices for O2C risk scoring

open_invoices = open_invoices.copy()

open_invoices["invoice_amount"] = open_invoices["total_open_amount"]

open_invoices["posting_to_due_days"] = (
    open_invoices["due_date"] - open_invoices["posting_date"]
).dt.days

open_invoices["due_month"] = open_invoices["due_date"].dt.month
open_invoices["due_day_of_week"] = open_invoices["due_date"].dt.dayofweek
open_invoices["posting_month"] = open_invoices["posting_date"].dt.month

# Attach strictly-prior customer history.
# Only closed invoices are used to construct this history.

open_invoices = open_invoices.merge(
    history,
    on=["cust_number", "due_date"],
    how="left",
    validate="many_to_one"
)

print("Open invoices:", len(open_invoices))

print("\nMissing values:")
print(open_invoices[feature_cols].isna().sum())

print("\nFeature preview:")
display(
    open_invoices[
        ["cust_number", "invoice_id", "due_date"] + feature_cols
    ].head()
)

Open invoices: 9681

Missing values:
invoice_amount                 0
posting_to_due_days            0
due_month                      0
due_day_of_week                0
posting_month                  0
business_code                  0
invoice_currency               0
cust_payment_terms             0
prior_invoice_count         9366
prior_late_rate             9366
prior_avg_days_late         9366
prior_median_days_late      9366
prior_avg_invoice_amount    9366
prior_max_days_late         9366
dtype: int64

Feature preview:


,cust_number,invoice_id,due_date,invoice_amount,posting_to_due_days,due_month,due_day_of_week,posting_month,business_code,invoice_currency,cust_payment_terms,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late
0,140105686,2.960623e+09,2020-04-10,3299.70,11,4,4,3,CA02,CAD,CA10,NaN,NaN,NaN,NaN,NaN,NaN
1,200744019,1.930659e+09,2020-04-03,11173.02,15,4,4,3,U001,USD,NAA8,NaN,NaN,NaN,NaN,NaN,NaN
2,200418007,1.930611e+09,2020-03-26,3525.59,15,3,3,3,U001,USD,NAA8,NaN,NaN,NaN,NaN,NaN,NaN
3,200739534,1.930788e+09,2020-04-30,121105.65,15,4,3,4,U001,USD,NAA8,NaN,NaN,NaN,NaN,NaN,NaN
4,200353024,1.930817e+09,2020-04-26,3726.06,3,4,6,4,U001,USD,NAM2,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Correct as-of customer history lookup for open invoices

# Keep one historical state per customer + due date
closed_history = (
    closed[
        [
            "cust_number",
            "due_date",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_median_days_late",
            "prior_avg_invoice_amount",
            "prior_max_days_late"
        ]
    ]
    .drop_duplicates(["cust_number", "due_date"])
    .sort_values(["due_date", "cust_number"])
    .reset_index(drop=True)
)

open_for_merge = (
    open_invoices
    .drop(
        columns=[
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "prior_median_days_late",
            "prior_avg_invoice_amount",
            "prior_max_days_late"
        ],
        errors="ignore"
    )
    .sort_values(["due_date", "cust_number"])
    .reset_index(drop=True)
)

open_invoices = pd.merge_asof(
    open_for_merge,
    closed_history,
    on="due_date",
    by="cust_number",
    direction="backward",
    allow_exact_matches=False
)

history_cols = [
    "prior_invoice_count",
    "prior_late_rate",
    "prior_avg_days_late",
    "prior_median_days_late",
    "prior_avg_invoice_amount",
    "prior_max_days_late"
]

# No prior invoice = legitimate zero-history customer
open_invoices[history_cols] = (
    open_invoices[history_cols].fillna(0)
)

print("Open invoices:", len(open_invoices))

print("\nMissing values:")
print(open_invoices[feature_cols].isna().sum())

print("\nInvoices with no prior closed history:")
print(
    (open_invoices["prior_invoice_count"] == 0).sum()
)

print("\nFeature preview:")
display(
    open_invoices[
        ["cust_number", "invoice_id", "due_date"] + feature_cols
    ].head(10)
)

Open invoices: 9681

Missing values:
invoice_amount              0
posting_to_due_days         0
due_month                   0
due_day_of_week             0
posting_month               0
business_code               0
invoice_currency            0
cust_payment_terms          0
prior_invoice_count         0
prior_late_rate             0
prior_avg_days_late         0
prior_median_days_late      0
prior_avg_invoice_amount    0
prior_max_days_late         0
dtype: int64

Invoices with no prior closed history:
120

Feature preview:


,cust_number,invoice_id,due_date,invoice_amount,posting_to_due_days,due_month,due_day_of_week,posting_month,business_code,invoice_currency,cust_payment_terms,prior_invoice_count,prior_late_rate,prior_avg_days_late,prior_median_days_late,prior_avg_invoice_amount,prior_max_days_late
0,CCU013,1.930570e+09,2020-02-27,14360.96,0,2,3,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
1,CCU013,1.930558e+09,2020-02-27,26226.85,0,2,3,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
2,CCU013,1.930570e+09,2020-02-28,37209.64,0,2,4,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
3,CCU013,1.930572e+09,2020-02-28,77.07,0,2,4,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
4,CCU013,1.930574e+09,2020-02-28,8905.00,0,2,4,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
5,CCU013,1.930560e+09,2020-02-28,5243.13,0,2,4,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
6,CCU013,1.930584e+09,2020-02-29,10044.00,0,2,5,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
7,CCU013,1.930570e+09,2020-02-29,3030.00,0,2,5,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
8,CCU013,1.930568e+09,2020-02-29,3462.95,0,2,5,2,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0
9,CCU013,1.930583e+09,2020-03-01,115417.74,0,3,6,3,U001,USD,NAX2,531.0,1.0,43.557692,42.0,14930.114673,204.0


In [38]:
# Score currently open invoices for late-payment risk

open_invoices["late_payment_probability"] = (
    o2c_model.predict_proba(
        open_invoices[feature_cols]
    )[:, 1]
)

print("Open invoices scored:", len(open_invoices))

print("\nProbability statistics:")
display(
    open_invoices["late_payment_probability"].describe()
)

print("\nHighest-risk open invoices:")
display(
    open_invoices[
        [
            "cust_number",
            "invoice_id",
            "due_date",
            "invoice_amount",
            "prior_invoice_count",
            "prior_late_rate",
            "prior_avg_days_late",
            "late_payment_probability"
        ]
    ]
    .sort_values(
        "late_payment_probability",
        ascending=False
    )
    .head(10)
)

Open invoices scored: 9681

Probability statistics:


,late_payment_probability
count,9681.000000
mean,0.430890
std,0.235131
min,0.092038
25%,0.238595
50%,0.303095
75%,0.658759
max,0.977620



Highest-risk open invoices:


,cust_number,invoice_id,due_date,invoice_amount,prior_invoice_count,prior_late_rate,prior_avg_days_late,late_payment_probability
3990,140106408,2.960626e+09,2020-04-11,62156.26,426.0,1.0,9.698658,0.977620
1926,140106408,2.960622e+09,2020-03-28,62882.86,426.0,1.0,9.698658,0.977219
2382,140106408,2.960622e+09,2020-04-01,80328.75,426.0,1.0,9.698658,0.976496
2674,140106408,2.960622e+09,2020-04-03,58649.11,426.0,1.0,9.698658,0.976481
4179,140106408,2.960626e+09,2020-04-12,63111.52,426.0,1.0,9.698658,0.976462
8098,140106408,2.960632e+09,2020-05-13,62235.98,426.0,1.0,9.698658,0.976445
708,140106408,2.960619e+09,2020-03-19,47072.24,426.0,1.0,9.698658,0.976378
707,140106408,2.960618e+09,2020-03-19,57833.84,426.0,1.0,9.698658,0.976378
62,140106408,2.960618e+09,2020-03-11,71300.71,423.0,1.0,9.698794,0.976221
3098,140106408,2.960624e+09,2020-04-06,63647.93,426.0,1.0,9.698658,0.975993


In [39]:
# Convert O2C probabilities into operational risk tiers

def risk_tier(probability):
    if probability >= 0.75:
        return "HIGH"
    elif probability >= 0.50:
        return "MEDIUM"
    else:
        return "LOW"

open_invoices["risk_tier"] = (
    open_invoices["late_payment_probability"]
    .apply(risk_tier)
)

print("O2C Risk Tier Distribution")
print("---------------------------")

tier_counts = (
    open_invoices["risk_tier"]
    .value_counts()
    .reindex(["HIGH", "MEDIUM", "LOW"])
    .fillna(0)
    .astype(int)
)

print(tier_counts)

print("\nPercent of open invoices:")
print(
    (tier_counts / len(open_invoices) * 100)
    .round(2)
)

O2C Risk Tier Distribution
---------------------------
risk_tier
HIGH      1203
MEDIUM    2359
LOW       6119
Name: count, dtype: int64

Percent of open invoices:
risk_tier
HIGH      12.43
MEDIUM    24.37
LOW       63.21
Name: count, dtype: float64


In [40]:
from pathlib import Path
import joblib

artifact_dir = Path("ml/o2c/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "selected_o2c_late_payment_model.pkl"

joblib.dump(o2c_model, model_path)

print(model_path)
print(model_path.exists())

ml/o2c/artifacts/selected_o2c_late_payment_model.pkl
True
